In [1]:
!pip install tensorflow keras opencv-python numpy matplotlib

**Import Libraries**

In [2]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator


**Data Preparation**(Images ko load krna)

**Data Augmentation**(Images ko rotate,zoom or flip krta hai taka model bhtr sekhy)

In [3]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2 # 20% data testing ke liye
)


**Load Training Data**

In [4]:
train_data = train_datagen.flow_from_directory(
    '/content/drive/MyDrive/data',
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary', # Kyun ke sirf 2 classes hain (Mask ya No Mask)
    subset='training'
)


Found 6069 images belonging to 2 classes.


**Load Validation Data**

In [5]:
validation_data = train_datagen.flow_from_directory(
    '/content/drive/MyDrive/data',
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)


Found 1516 images belonging to 2 classes.


**Building the CNN Model**

In [6]:
model = Sequential([
    # 1st Convolutional Layer (Features nikalne ke liye)
    Conv2D(32, (3,3), activation='relu', input_shape=(150, 150, 3)),
    MaxPooling2D(2,2), # Image size ko chota karne ke liye

    # 2nd Convolutional Layer
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),


    # 3rd Convolutional Layer
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    # convert data to 1D array
    Flatten(),

    # Hidden Layer (Sochne aur samajhne ke liye)
    Dense(128, activation='relu'),
    Dropout(0.5), # Overfitting se bachane ke liye kuch neurons ko band karna

    # Output Layer (0 ya 1 result dega: Mask / No Mask)
    Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


**Compile the Model**

In [7]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])


**Train the model**

In [8]:
#epochs=1 mean Model 1  dafa poore data ko dekhega
model.fit(train_data, validation_data=validation_data, epochs=1)


125/190 ━━━━━━━━━━━━━━━━━━━━ 5:57 6s/step - accuracy: 0.7102 - loss: 0.5747

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


190/190 ━━━━━━━━━━━━━━━━━━━━ 1320s 7s/step - accuracy: 0.8276 - loss: 0.3949 - val_accuracy: 0.9077 - val_loss: 0.2359


**Save the model**

In [9]:
model.save('face_mask_model.h5')


In [10]:
print("Model Save ho gaya hai!")

Model Save ho gaya hai!


In [ ]:
%%writefile app.py
import streamlit as st
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# Set page title and layout
st.set_page_config(page_title="Face Mask Detector", page_icon="😷", layout="centered")

# Efficiently load the model and cascade once
@st.cache_resource
def load_resources():
    model = load_model('face_mask_model.h5')
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    return model, face_cascade

try:
    model, face_cascade = load_resources()
    st.title("😷 Face Mask Detection System")
    st.info("Take a photo to check if you are wearing a mask.")

    # Streamlit Camera Input - very fast in browsers
    img_file_buffer = st.camera_input("")

    if img_file_buffer is not None:
        # Convert buffer to OpenCV image
        file_bytes = np.asarray(bytearray(img_file_buffer.read()), dtype=np.uint8)
        image = cv2.imdecode(file_bytes, 1)
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # Detect faces
        faces = face_cascade.detectMultiScale(gray, 1.1, 4)

        if len(faces) == 0:
            st.warning("No face detected!")
        else:
            for (x, y, w, h) in faces:
                # Extract and prepare face for model
                face = image[y:y+h, x:x+w]
                face = cv2.resize(face, (150, 150)) / 255.0
                face = np.expand_dims(face, axis=0)

                # Prediction
                prediction = model.predict(face, verbose=0) # verbose=0 makes it faster/cleaner

                if prediction[0][0] < 0.5:
                    label, color = "Mask Detected", (0, 255, 0)
                    st.success(f"✅ {label}")
                else:
                    label, color = "No Mask", (255, 0, 0)
                    st.error(f"❌ {label}")

                # Annotate image
                cv2.rectangle(image, (x, y), (x+w, y+h), color, 3)
                cv2.putText(image, label, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

            # Convert BGR to RGB for Streamlit display
            st.image(cv2.cvtColor(image, cv2.COLOR_BGR2RGB), use_container_width=True)

except Exception as e:
    st.error(f"Error: Ensure 'face_mask_model.h5' is in the project folder. ({e})")

### How to run the Streamlit app in Colab
To see your app, you need to install streamlit and localtunnel, then run the server:

1. `!pip install streamlit`
2. `!npm install -g localtunnel`
3. `!streamlit run app.py & npx localtunnel --port 8501`

Then click the link provided by localtunnel.